# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  

**`Student Name`:** Arnav Mehta 

**`Roll Number`:**  U20230130

**`GitHub Branch`:** arnav_U20230130  

# Imports and Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
!pip install rlcmab_sampler
from rlcmab_sampler import sampler


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 2.9 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 4.4 MB/s eta 0:00:00 MB/s eta 0:00:01:02
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: scipy━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [numpy]
    Found existing installation: scipy 1.16.2━━━━━ 0/3 [numpy]
    Uninstalling scipy-1.16.2:━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [scipy]
      Successfully uninstalled scipy-1.16.27m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [scipy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [rlcmab_sampler] 1/3 [scipy]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.2 which is inc

# Load Datasets

In [3]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [7]:
# Data Pre-processing for users and articles

print(f"""
Missing values in news_df:
{news_df.isnull().sum()}

Missing values in train_users:
{train_users.isnull().sum()}

Missing values in test_users:
{test_users.isnull().sum()}
""")

news_df_clean = news_df.dropna().reset_index(drop=True)
train_users_clean = train_users.dropna().reset_index(drop=True)
test_users_clean = test_users.dropna().reset_index(drop=True)

print(f"""
news_df_clean shape: {news_df_clean.shape}
train_users_clean shape: {train_users_clean.shape}
test_users_clean shape: {test_users_clean.shape}
""")



Missing values in news_df:
link                     0
headline                 6
category                 0
short_description    19712
authors              37418
date                     0
dtype: int64

Missing values in train_users:
user_id            0
age                0
income             0
clicks             0
purchase_amount    0
label              0
dtype: int64

Missing values in test_users:
user_id            0
age                0
income             0
clicks             0
purchase_amount    0
label              0
dtype: int64


news_df_clean shape: (156859, 6)
train_users_clean shape: (2000, 6)
test_users_clean shape: (2000, 6)



In [8]:
label_encoder = LabelEncoder()
train_users_clean["label_encoded"] = label_encoder.fit_transform(train_users_clean["label"])
test_users_clean["label_encoded"] = label_encoder.transform(test_users_clean["label"])

news_category_encoder = LabelEncoder()
news_df_clean["category_encoded"] = news_category_encoder.fit_transform(news_df_clean["category"])

feature_cols = ["age", "income", "clicks", "purchase_amount"]

X_train = train_users_clean[feature_cols]
y_train = train_users_clean["label_encoded"]

X_test = test_users_clean[feature_cols]
y_test = test_users_clean["label_encoded"]
print()
print("Feature columns used:", feature_cols)
print("X_train shape:", X_train.shape, ", y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape, ", y_test shape:", y_test.shape)


Feature columns used: ['age', 'income', 'clicks', 'purchase_amount']
X_train shape: (2000, 4) , y_train shape: (2000,)
X_test shape: (2000, 4) , y_test shape: (2000,)


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [16]:
# User Classification Model (Context Detector)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Multinomial logistic regression to predict user segment (User1/User2/User3)
user_classifier = LogisticRegression(max_iter=10000)
user_classifier.fit(X_train, y_train)

# Evaluate on held-out test set
y_pred = user_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy of User Classifier: {accuracy:.3f}")
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Keep references for later contextual bandit section
user_context_clf = user_classifier
user_label_encoder = label_encoder
user_feature_cols = feature_cols

Test Accuracy of User Classifier: 0.329
Classification Report:

              precision    recall  f1-score   support

       user1       0.35      0.50      0.41       672
       user2       0.32      0.36      0.34       679
       user3       0.29      0.12      0.17       649

    accuracy                           0.33      2000
   macro avg       0.32      0.33      0.31      2000
weighted avg       0.32      0.33      0.31      2000



# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
